# 🎭 Notebook 1 — Analyse de Sentiment
**ENSA Al Hoceima — Mini-Projet NLP | Ingénierie des Données 2A**

---

## Objectif
Analyser le sentiment (POSITIF / NÉGATIF) de textes en utilisant :
- **Modèle** : `distilbert-base-uncased-finetuned-sst-2-english` (DistilBERT fine-tuné SST-2)
- **Dataset** : `SST-2` (Stanford Sentiment Treebank — critiques de films)
- **Bonus** : `nlptown/bert-base-multilingual-uncased-sentiment` pour le français

## Pipeline
```
Texte brut → Tokenizer → DistilBERT → Softmax → POSITIVE / NEGATIVE
```

## 1. Imports et Configuration

In [ ]:
import sys
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from collections import Counter

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    pipeline,
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='husl')

# ─── Chemins du projet ───────────────────────────────────────────────────────
PROJECT_ROOT = Path().resolve().parents[0]          # racine du projet
DATA_DIR     = PROJECT_ROOT / 'data' / 'raw' / 'sst2'
MODEL_PATH   = PROJECT_ROOT / 'models' / 'sentiment' / 'distilbert-sentiment'
MODEL_PATH_ML= PROJECT_ROOT / 'models' / 'sentiment' / 'bert-multilingual-sentiment'

DEVICE = 0 if torch.cuda.is_available() else -1
print(f"✅ Device : {'GPU (CUDA)' if DEVICE == 0 else 'CPU'}")
print(f"📁 Modèle : {MODEL_PATH}")
print(f"📁 Data   : {DATA_DIR}")

## 2. Chargement du Dataset SST-2

In [ ]:
# ─── Chargement depuis les fichiers JSON sauvegardés par load_datasets.py ───
def charger_sst2(split='train', n=500):
    path = DATA_DIR / f'{split}.json'
    if not path.exists():
        # Fallback : chargement direct HuggingFace
        print(f"⚠️  Fichier local non trouvé → chargement HuggingFace")
        from datasets import load_dataset
        ds = load_dataset('sst2', split=split)
        df = pd.DataFrame({'sentence': ds['sentence'], 'label': ds['label']})
    else:
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        df = pd.DataFrame(data)
    return df.head(n)

df_train = charger_sst2('train', n=500)
df_val   = charger_sst2('validation', n=200)

print(f"\n📊 Dataset SST-2 chargé :")
print(f"   Train      : {len(df_train)} exemples")
print(f"   Validation : {len(df_val)} exemples")
print(f"\n🔍 Aperçu :")
df_train.head(5)

In [ ]:
# ─── Distribution des classes ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (df, title) in zip(axes, [(df_train, 'Train'), (df_val, 'Validation')]):
    counts = df['label'].value_counts()
    labels_text = ['NEGATIVE (0)', 'POSITIVE (1)']
    colors = ['#e74c3c', '#2ecc71']
    bars = ax.bar([labels_text[i] for i in counts.index], counts.values, color=colors, edgecolor='white', linewidth=1.5)
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3, str(val),
                ha='center', fontweight='bold', fontsize=11)
    ax.set_title(f'Distribution — {title}', fontsize=13, fontweight='bold')
    ax.set_ylabel('Nombre d\'exemples')
    ax.set_ylim(0, counts.max() * 1.15)

plt.suptitle('SST-2 : Distribution des sentiments', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('sst2_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print("\n📈 Distribution des labels :")
print(df_train['label'].value_counts())

## 3. Exploration du Tokenizer BERT

In [ ]:
# ─── Chargement du tokenizer ─────────────────────────────────────────────────
if MODEL_PATH.exists():
    tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH))
    print(f"✅ Tokenizer chargé depuis : {MODEL_PATH}")
else:
    tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased-finetuned-sst-2-english')
    print("✅ Tokenizer chargé depuis HuggingFace")

# ─── Démonstration du tokenizer ──────────────────────────────────────────────
exemple = "This movie was absolutely fantastic! The acting was superb."
tokens = tokenizer(exemple, return_tensors='pt')

print(f"\n📝 Phrase originale :")
print(f"   '{exemple}'")
print(f"\n🔤 Tokens IDs : {tokens['input_ids'][0].tolist()}")
print(f"🔤 Tokens     : {tokenizer.convert_ids_to_tokens(tokens['input_ids'][0])}")
print(f"📏 Longueur   : {tokens['input_ids'].shape[1]} tokens")
print(f"📌 Vocabulaire du tokenizer : {tokenizer.vocab_size:,} tokens")

In [ ]:
# ─── Analyse des longueurs de séquences dans SST-2 ───────────────────────────
longueurs = []
for phrase in df_train['sentence'].head(200):
    toks = tokenizer(str(phrase), truncation=True, max_length=128)
    longueurs.append(len(toks['input_ids']))

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(longueurs, bins=30, color='#3498db', edgecolor='white', alpha=0.85)
ax.axvline(np.mean(longueurs), color='red', linestyle='--', linewidth=2,
           label=f'Moyenne : {np.mean(longueurs):.1f}')
ax.axvline(np.median(longueurs), color='orange', linestyle='--', linewidth=2,
           label=f'Médiane : {np.median(longueurs):.1f}')
ax.set_title('Distribution des longueurs de séquences (SST-2)', fontsize=13, fontweight='bold')
ax.set_xlabel('Nombre de tokens')
ax.set_ylabel('Fréquence')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('sst2_longueurs.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Statistiques : min={min(longueurs)}, max={max(longueurs)}, mean={np.mean(longueurs):.1f}")

## 4. Pipeline de Sentiment — DistilBERT

In [ ]:
# ─── Chargement du pipeline ───────────────────────────────────────────────────
model_source = str(MODEL_PATH) if MODEL_PATH.exists() else 'distilbert-base-uncased-finetuned-sst-2-english'
print(f"⏳ Chargement du pipeline sentiment...")

pipe_sentiment = pipeline(
    task='sentiment-analysis',
    model=model_source,
    tokenizer=model_source,
    device=DEVICE,
    truncation=True,
    max_length=512,
)
print("✅ Pipeline DistilBERT-SST2 prêt !")

In [ ]:
# ─── Test sur des phrases d'exemple ──────────────────────────────────────────
phrases_test = [
    "This film is a masterpiece. Absolutely brilliant performance!",
    "The worst movie I have ever seen. Total waste of time.",
    "It was okay, nothing special but not terrible either.",
    "An emotional rollercoaster that left me speechless.",
    "Boring, predictable, and poorly acted. Very disappointing.",
    "I loved every single minute. Highly recommend!",
    "The plot was confusing and the characters were flat.",
    "A delightful surprise — fresh, funny, and heartfelt.",
]

resultats = pipe_sentiment(phrases_test)

print("\n🎭 ANALYSE DE SENTIMENT — RÉSULTATS\n")
print(f"{'Phrase':<55} {'Label':<12} {'Score':>6}")
print('─' * 75)
for phrase, res in zip(phrases_test, resultats):
    emoji = '🟢' if res['label'] == 'POSITIVE' else '🔴'
    phrase_short = phrase[:52] + '...' if len(phrase) > 52 else phrase
    print(f"{phrase_short:<55} {emoji} {res['label']:<10} {res['score']:.4f}")

In [ ]:
# ─── Visualisation des scores de confiance ───────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))

couleurs = ['#2ecc71' if r['label'] == 'POSITIVE' else '#e74c3c' for r in resultats]
scores   = [r['score'] for r in resultats]
labels_r = [f"{r['label']} ({r['score']:.3f})" for r in resultats]
x_labels = [p[:35] + '...' if len(p) > 35 else p for p in phrases_test]

bars = ax.barh(range(len(scores)), scores, color=couleurs, edgecolor='white', linewidth=1)
ax.set_yticks(range(len(phrases_test)))
ax.set_yticklabels(x_labels, fontsize=9)
ax.set_xlim(0, 1.15)
ax.axvline(0.5, color='gray', linestyle='--', linewidth=1.5, alpha=0.7, label='Seuil 0.5')
for i, (bar, score) in enumerate(zip(bars, scores)):
    ax.text(score + 0.01, bar.get_y() + bar.get_height()/2,
            f'{score:.3f}', va='center', fontweight='bold', fontsize=9)

pos_patch = mpatches.Patch(color='#2ecc71', label='POSITIVE')
neg_patch = mpatches.Patch(color='#e74c3c', label='NEGATIVE')
ax.legend(handles=[pos_patch, neg_patch], loc='lower right', fontsize=10)
ax.set_xlabel('Score de confiance', fontsize=11)
ax.set_title('DistilBERT-SST2 : Scores de confiance par phrase', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('sentiment_scores.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Évaluation sur le Jeu de Validation SST-2

In [ ]:
# ─── Évaluation sur 100 exemples de validation ───────────────────────────────
N_EVAL = 100
df_eval = df_val.head(N_EVAL).copy()
phrases_eval = df_eval['sentence'].astype(str).tolist()
labels_vrais = df_eval['label'].tolist()  # 0 = NEGATIVE, 1 = POSITIVE

print(f"⏳ Inférence sur {N_EVAL} exemples...")
predictions = pipe_sentiment(phrases_eval, batch_size=16, truncation=True)

# Conversion label → int
label_map = {'POSITIVE': 1, 'NEGATIVE': 0}
preds_int  = [label_map[p['label']] for p in predictions]
scores_conf = [p['score'] for p in predictions]

# Métriques
correct = sum(p == v for p, v in zip(preds_int, labels_vrais))
accuracy = correct / N_EVAL

print(f"\n📊 RÉSULTATS D'ÉVALUATION ({N_EVAL} exemples)")
print(f"   Accuracy          : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"   Corrects          : {correct}/{N_EVAL}")
print(f"   Score moyen       : {np.mean(scores_conf):.4f}")

In [ ]:
# ─── Matrice de confusion ────────────────────────────────────────────────────
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(labels_vrais, preds_int)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['NEGATIVE', 'POSITIVE'],
            yticklabels=['NEGATIVE', 'POSITIVE'],
            linewidths=0.5)
ax.set_xlabel('Prédiction', fontsize=12)
ax.set_ylabel('Vérité terrain', fontsize=12)
ax.set_title(f'Matrice de confusion — DistilBERT-SST2\nAccuracy : {accuracy:.2%}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix_sentiment.png', dpi=120, bbox_inches='tight')
plt.show()

print("\n📋 Rapport de classification :")
print(classification_report(labels_vrais, preds_int, target_names=['NEGATIVE', 'POSITIVE']))

## 6. Bonus — Modèle Multilingue (Français / Arabe translittéré)

In [ ]:
# ─── Pipeline multilingue (1–5 étoiles) ─────────────────────────────────────
ml_source = str(MODEL_PATH_ML) if MODEL_PATH_ML.exists() else 'nlptown/bert-base-multilingual-uncased-sentiment'
pipe_ml = pipeline(
    task='sentiment-analysis',
    model=ml_source,
    tokenizer=ml_source,
    device=DEVICE,
    truncation=True,
)
print("✅ Pipeline multilingue prêt !")

phrases_fr = [
    "Ce film est absolument magnifique, un chef-d'œuvre!",
    "Quelle déception, l'histoire est ennuyeuse et mal jouée.",
    "Pas mal, sans plus. Quelques bons moments mais globalement ordinaire.",
    "J'ai adoré chaque scène, les acteurs sont extraordinaires!",
    "Le pire film que j'aie jamais vu. À éviter absolument.",
]

res_ml = pipe_ml(phrases_fr)

print("\n🌍 SENTIMENT MULTILINGUE (Français)\n")
print(f"{'Phrase':<52} {'Label':<12} {'Score':>6}")
print('─' * 72)
etoiles = {'1 star':'⭐','2 stars':'⭐⭐','3 stars':'⭐⭐⭐',
           '4 stars':'⭐⭐⭐⭐','5 stars':'⭐⭐⭐⭐⭐'}
for phrase, res in zip(phrases_fr, res_ml):
    phrase_s = phrase[:49] + '...' if len(phrase) > 49 else phrase
    stars = etoiles.get(res['label'], res['label'])
    print(f"{phrase_s:<52} {stars:<12} {res['score']:.4f}")

## 7. Analyse Approfondie — Interprétabilité

In [ ]:
# ─── Analyse des erreurs ─────────────────────────────────────────────────────
erreurs = [(phrases_eval[i], labels_vrais[i], preds_int[i], scores_conf[i])
           for i in range(N_EVAL) if preds_int[i] != labels_vrais[i]]

print(f"\n❌ ANALYSE DES ERREURS ({len(erreurs)} cas)\n")
label_names = {0: 'NEGATIVE', 1: 'POSITIVE'}
for phrase, vrai, pred, conf in erreurs[:8]:
    phrase_s = phrase[:65] + '...' if len(phrase) > 65 else phrase
    print(f"  Phrase   : {phrase_s}")
    print(f"  Vrai     : {label_names[vrai]}  |  Prédit : {label_names[pred]}  |  Confiance : {conf:.3f}")
    print()

In [ ]:
# ─── Distribution des scores de confiance par classe ─────────────────────────
pos_scores = [scores_conf[i] for i in range(N_EVAL) if preds_int[i] == 1]
neg_scores = [scores_conf[i] for i in range(N_EVAL) if preds_int[i] == 0]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(pos_scores, bins=20, alpha=0.7, color='#2ecc71', label=f'POSITIVE (n={len(pos_scores)})', edgecolor='white')
ax.hist(neg_scores, bins=20, alpha=0.7, color='#e74c3c', label=f'NEGATIVE (n={len(neg_scores)})', edgecolor='white')
ax.set_title('Distribution des scores de confiance par classe prédite', fontsize=13, fontweight='bold')
ax.set_xlabel('Score de confiance')
ax.set_ylabel('Fréquence')
ax.legend(fontsize=11)
ax.axvline(0.9, color='black', linestyle=':', alpha=0.7, label='0.9')
plt.tight_layout()
plt.savefig('confidence_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"\n📊 Résumé :")
print(f"   POSITIVE — moyenne confiance : {np.mean(pos_scores):.4f}")
print(f"   NEGATIVE — moyenne confiance : {np.mean(neg_scores):.4f}")

## 8. Résumé et Conclusion

In [ ]:
print("="*60)
print("  RÉSUMÉ — ANALYSE DE SENTIMENT")
print("="*60)
print(f"  Modèle utilisé    : DistilBERT-SST2")
print(f"  Dataset           : SST-2 (Stanford Sentiment Treebank)")
print(f"  Accuracy (val)    : {accuracy:.2%}")
print(f"  Erreurs           : {N_EVAL - correct}/{N_EVAL}")
print(f"  Device            : {'GPU' if DEVICE == 0 else 'CPU'}")
print()
print("  Points clés :")
print("  • DistilBERT est 60% plus rapide que BERT tout en conservant")
print("    97% de ses performances sur les tâches NLP")
print("  • Le tokenizer [CLS] capture le sentiment global de la phrase")
print("  • Les scores > 0.9 indiquent une très haute confiance du modèle")
print("  • Le modèle multilingue BERT supporte 6 langues dont le français")
print("="*60)